
# Native NeoOLAF DocRED batch run and evaluation — v6

This notebook freezes the successful **v5.1 Skai TV scientific configuration**
and applies it unchanged to a DocRED JSONL corpus.

The intended workflow is:

1. leave `RUN_ALL_DOCUMENTS = False` and run the first five documents;
2. inspect the aggregate and per-document evaluation;
3. change only `RUN_ALL_DOCUMENTS = True`;
4. rerun the batch cell to process every record in the JSONL.

Completed document runs are resumed. The first five documents are therefore not
paid for or executed again when the full-corpus run starts.

## Scientific constraints

- Native NeoOLAF Layers 0–12 run for every document.
- No file under `src/neoolaf` is modified.
- Gold entities and relations are removed before pipeline execution.
- The frozen v5.1 profile, user guidance, task guidance, ontology, and projection
  rules are identical for every document.
- No direct DocRED extractor, source-entity anchoring, gold-pair hints, or
  post-run relation invention is used.
- Gold data is loaded only after a document run for evaluation.
- Each document keeps its complete states, prompts, raw responses, logs,
  errors, ontology retrieval records, and analysis artifacts.
- Document-level launches use separate processes. Layer-level relation
  enrichment keeps its internal parallel workers.


In [ ]:

from __future__ import annotations

import json
import multiprocessing as mp
import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_batch_v6 import (
    BatchRunConfig,
    aggregate_batch_results,
    iter_jsonl,
    read_json,
    run_batch,
)

mp.freeze_support()
print("PROJECT_ROOT =", PROJECT_ROOT)


## 1. Corpus path and the one-variable smoke/full switch

In [ ]:

# Change only this variable after the five-document smoke run succeeds.
RUN_ALL_DOCUMENTS = False

# When RUN_ALL_DOCUMENTS=False, process exactly the first five matching records.
SMOKE_DOCUMENT_LIMIT = 5

# None means the entire JSONL regardless of its `type`/`split` field.
# Set to "dev" only when a split filter is scientifically intended.
TYPE_FILTER = None
START_INDEX = 0

# This is the full DocRED JSONL used by the earlier RAGTree notebooks.
DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# The same root is used for smoke and full runs. When RUN_ALL_DOCUMENTS becomes
# True, the first five completed document directories are detected and skipped.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_batch"

print("RUN_ALL_DOCUMENTS =", RUN_ALL_DOCUMENTS)
print("DATASET_JSONL =", DATASET_JSONL)
print("BATCH_ROOT =", BATCH_ROOT)


## 2. Frozen v5.1 scientific configuration and parallelism

In [ ]:

ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Parallel native NeoOLAF launches. Keep this between 3 and 5 as requested.
DOCUMENT_WORKERS = 4

# Internal v5.1 Layer 2 workers retained per document.
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

# Safe large-run behavior.
RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1 if not RUN_ALL_DOCUMENTS else 10

# The maximum theoretical burst is DOCUMENT_WORKERS × LAYER_WORKERS.
# Lower DOCUMENT_WORKERS first if OpenRouter returns HTTP 429.
print("Document workers:", DOCUMENT_WORKERS)
print("Layer workers per document:", LAYER_WORKERS)
print("Maximum theoretical concurrent Layer 2 requests:", DOCUMENT_WORKERS * LAYER_WORKERS)
print("API key available:", bool(API_KEY))


## 3. Preflight: files, corpus size, and anti-leakage assertions

In [ ]:

required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

total_records = 0
matching_records = 0
first_ids = []
for record in iter_jsonl(DATASET_JSONL):
    total_records += 1
    actual_type = str(record.get("type") or record.get("split") or "")
    matches = TYPE_FILTER is None or TYPE_FILTER in {"", "all", "*"} or actual_type == TYPE_FILTER
    if matches:
        matching_records += 1
        if len(first_ids) < 5:
            first_ids.append(record.get("document_id") or record.get("id") or record.get("title"))

selected_count = matching_records if RUN_ALL_DOCUMENTS else min(SMOKE_DOCUMENT_LIMIT, matching_records)

print("JSONL records:", total_records)
print("Matching records:", matching_records)
print("Selected this invocation:", selected_count)
print("First selected IDs:", first_ids)
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert 3 <= DOCUMENT_WORKERS <= 5

display(Markdown(
    f"**Mode:** {'full JSONL' if RUN_ALL_DOCUMENTS else 'five-document smoke'}  \\n"
    f"**Selected:** {selected_count} documents  \\n"
    f"**Resume:** {RESUME_COMPLETED}. The same batch root is retained across both modes."
))



## 4. Run native NeoOLAF in parallel

Every document receives:

- one no-gold input JSONL;
- one separate gold JSONL used only after execution;
- a unique run directory;
- full Layer 0–12 checkpoints and artifacts;
- raw LLM responses and prompt audits;
- ontology-retrieval logs;
- per-document strict relation and entity evaluation.

The parent process writes `batch_events.jsonl` while documents finish. A failed
document does not terminate the other launches. Transient failures are retried
once by default.


In [ ]:

RUN_PIPELINE = True

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    batch = run_batch(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=RUN_ALL_DOCUMENTS,
        smoke_document_limit=SMOKE_DOCUMENT_LIMIT,
        type_filter=TYPE_FILTER,
        start_index=START_INDEX,
        config=config,
        api_key=API_KEY,
    )
    print("Batch execution and aggregate evaluation completed.")
else:
    results = read_json(BATCH_ROOT / "document_results.json")
    batch = aggregate_batch_results(
        results=results,
        batch_root=BATCH_ROOT,
        relation_catalog_path=RELATION_CATALOG,
    )
    print("RUN_PIPELINE=False: aggregate evaluation rebuilt from saved results.")


## 5. Aggregate relation and entity evaluation

In [ ]:

summary = batch["summary"]
display(pd.DataFrame([{
    "documents_requested": summary["documents_requested"],
    "documents_completed_or_resumed": summary["documents_completed_or_resumed"],
    "documents_failed": summary["documents_failed"],
    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],
    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],
    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],
    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],
    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}]))

print("First-failure counts across completed documents:")
display(pd.DataFrame(
    sorted(summary["first_failure_counts"].items(), key=lambda item: (-item[1], item[0])),
    columns=["first_failure", "gold_relations"],
))


## 6. Per-document results

In [ ]:

per_document = pd.DataFrame(batch["per_document"])
if not per_document.empty:
    display(per_document.sort_values(
        ["relation_f1", "relation_recall", "relation_precision"],
        ascending=[True, True, True],
    ))
    display(per_document[[
        "relation_precision", "relation_recall", "relation_f1",
        "entity_precision", "entity_recall", "entity_f1",
        "pipeline_seconds",
    ]].describe())
else:
    print("No completed documents.")


## 7. Per-relation micro evaluation

In [ ]:

per_relation = pd.DataFrame(batch["per_relation"])
if not per_relation.empty:
    display(per_relation.sort_values(
        ["gold", "f1", "recall"],
        ascending=[False, True, True],
    ))
else:
    print("No relation metrics available.")


## 8. Aggregate cumulative layer contribution

In [ ]:

cumulative = pd.DataFrame(batch["cumulative"])
display(cumulative)

if not cumulative.empty:
    display(cumulative[[
        "layer_index", "layer_name", "predicted", "gold",
        "true_positive", "false_positive", "false_negative",
        "precision", "recall", "f1",
    ]])


## 9. Failures, resume state, and saved outputs

In [ ]:

failed = pd.DataFrame(batch["failed"])
print("Failed documents:", len(failed))
if not failed.empty:
    display(failed[[
        column for column in [
            "selection_index", "document_id", "title", "status",
            "error_type", "error", "run_dir",
        ]
        if column in failed.columns
    ]])

print("Batch output root:", BATCH_ROOT)
print("Aggregate summary:", BATCH_ROOT / "aggregate_analysis/batch_summary.json")
print("Per-document CSV:", BATCH_ROOT / "aggregate_analysis/per_document_metrics.csv")
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv")
print("Predictions JSONL:", BATCH_ROOT / "aggregate_analysis/predictions.jsonl")
print("Failures JSONL:", BATCH_ROOT / "aggregate_analysis/failed_documents.jsonl")
print("Progress events:", BATCH_ROOT / "batch_events.jsonl")
print("Selection manifest:", BATCH_ROOT / "selection_manifest.json")



## 10. Full-corpus launch

After the five-document result is acceptable:

```python
RUN_ALL_DOCUMENTS = True
```

Rerun the configuration, preflight, and batch cells. Do not change the batch
root. The first five scientific fingerprints match and their completed runs are
loaded rather than executed again.

For HTTP 429 errors, reduce `DOCUMENT_WORKERS` from 4 to 3 before reducing the
internal `LAYER_WORKERS`. This preserves most of the per-document v5.1 speedup.
